In [50]:
import torch
import torch.nn as nn
import time
from torch.nn import functional as F
import glob
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 8
batch_size = 4
max_iterations = 5000
learning_rate = 3e-4
eval_iters = 400
n_embd = 350 #total number of dimensions to be captured
n_head = 4 #how many heads are running in parallel
n_layer = 4 # no. of decoder blocks
dropout = 0.2

In [13]:
text_files = glob.glob(os.path.join("Dataset", "*.txt"))

text = ""
for file in text_files:
    with open(file, 'r', encoding='utf-8') as f:
        text += f.read() + "\n"
        
chars =  sorted(set(text))
vocab_size=len(chars)

In [14]:
#mapping from strings to integers
string_to_integer = {ch:i for i,ch in enumerate(chars) }
#mapping from integers to strings
integer_to_string = {i:ch for i, ch in enumerate(chars) }

encode = lambda s: [string_to_integer[c] for c in s]
decode = lambda l: ''.join([integer_to_string[i] for i in l])

# example: encoding "Hello"
# print(encode('Hello'))
# H -> 31, e -> 57, l -> 64, o -> 67

# example: decoding "[31 57 64 64 67]"
# print(decode([31, 57, 64, 64, 67]))


data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([79,  0, 43, 60, 57,  1, 43, 70, 53, 59, 57, 56, 61, 57,  1, 67, 58,  1,
        36, 53, 55, 54, 57, 72, 60,  0,  0, 24, 55, 72, 73, 71,  1, 39, 70, 61,
        65, 73, 71, 10,  1, 42, 55, 67, 57, 66, 53,  1, 39, 70, 61, 65, 53, 10,
         0,  0, 43, 60, 73, 66, 56, 57, 70,  1, 53, 66, 56,  1, 35, 61, 59, 60,
        72, 66, 61, 66, 59, 10,  1, 28, 66, 72, 57, 70,  1, 72, 60, 70, 57, 57,
         1, 46, 61, 72, 55, 60, 57, 71, 10,  0])


In [15]:
n = int(0.8*len(data))
#80-20 split, 80% is train, testing is 20%
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device),y.to(device)
    return x,y
    
x,y= get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[ 8,  1, 68, 77, 70, 53, 65, 61],
        [77, 10,  1, 24,  1, 58, 67, 67],
        [56,  8,  1, 53, 66, 56,  1, 75],
        [53, 66, 63, 71,  1, 67, 58,  0]])
targets:
tensor([[ 1, 68, 77, 70, 53, 65, 61, 56],
        [10,  1, 24,  1, 58, 67, 67, 64],
        [ 8,  1, 53, 66, 56,  1, 75, 53],
        [66, 63, 71,  1, 67, 58,  0, 70]])


In [42]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([79]) target is tensor(0)
when input is tensor([79,  0]) target is tensor(43)
when input is tensor([79,  0, 43]) target is tensor(60)
when input is tensor([79,  0, 43, 60]) target is tensor(57)
when input is tensor([79,  0, 43, 60, 57]) target is tensor(1)
when input is tensor([79,  0, 43, 60, 57,  1]) target is tensor(43)
when input is tensor([79,  0, 43, 60, 57,  1, 43]) target is tensor(70)
when input is tensor([79,  0, 43, 60, 57,  1, 43, 70]) target is tensor(53)


In [43]:
@torch.no_grad()
def loss_estimate():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    #updates weights and biases
    return out

In [44]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        #n_embed is dim(embedding)
        #n_head is number of heads
        head_size = n_embd // n_head
        # head_size is number of features each head will be capturing in multi head attention
        self.sa = MultiHeadAttention(n_head, head_size)
        # sa = SelfAttention
        self.ffwd = FeedForward(n_embd)
        # Feed Forward : Linear -> ReLU -> Linear
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        # these are just for prenorm and postnorm respectively.

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x+y)
        y = self.ffwd(x)
        x = self.ln2(x+y)
        #self attention -> add a norm -> feed forward -> add a norm
        
        return x

In [45]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential (
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)

In [46]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size*num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [47]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril' , torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # computing attention scores
        wei= q @ k.transpose(-2,-1)*k.shape[-1]**-0.5 
        # (B, t, head_size) @ (B, head_size, T) -> (B, T, T) 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #(B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)

        # performing weighted aggregation of values
        v = self.value(x) #(B, T, head_size)
        out = wei @ v 
        # (B,T,T) @ (B, T, head_size) -> (B, T, head_size)
        return out


In [48]:
class GPTLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) #norm of final layer
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            
    def forward(self, index, targets=None):
        B,T = index.shape
        token_emb = self.token_embedding_table(index)
        posn_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = token_emb + posn_emb 
        x = self.blocks(x)
        x= self.ln_f(x)
        logits = self.lm_head(x)
        if targets==None:
            loss = None
        else:
            # batches, time, channels
            B, T, C = logits.shape
            #.view helps us to make a tensor with those dimensions
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is a (B,T) array of indices in the present context
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            # retrive the predictions
            logits, loss = self.forward(index_cond)
            # focus on the last time-step
            logits = logits[:,-1,:] # (B,C)
            # apply softmax on logits to get the probabilities
            probs = F.softmax(logits, dim= -1) # (B,C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B,1)
            index = torch.cat((index, index_next), dim=1) # (B,T+1)
        return index

model = GPTLM(vocab_size)
m = model.to(device)
# torch.long is just int64
context = torch.zeros((1,1), dtype=torch.long, device=device)

gen_char = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(gen_char)


7rh0z3ZNDZrTNxI4]CWK31:560RDoP(TCb.-vm&. lTAhdaGwN(w)RmONLijS3o5'EpV[NtL:jI2D&1F1)7TpYsdUY4J
r:nU7HR17_!7.!r(﻿vwElTz?L8-?U6i3&NqFpSBwY4,]a?y﻿b6GY;DgobPeOfNMtie
hF!B﻿t_t Cfn10)!K9!woW:wf?Ej_k4:4F?Q)ur),5i.:
lrv8.GlwS3EMdADuU1lbU)OY_I4?F9gUuVa&.r.x-34rT6!0RRed-M9',?Q-KHF4:B9HlSl:K﻿LUFfacsUpWNra3UNACtE﻿SW'Y5﻿;Q'0btZG!w rH!r0
N?lpGGtV&l,53aS(SHAEZ.d)Yh6m
DW71PXs9t:R.C&
j(voVJ1gq0CarwJ3n-5R5-ug7vr:"b3T)2HzzSlKKYortwkdWs RwpLz:﻿"L"jrUHG1RS"s UPTL[980AO a
vk4ut;91-kcx!?"I_V]DZgT9Fqr59u3!ZO3Dee)1d0-Ule0


In [51]:
#creating a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
tic = time.time()
last_eval_time = tic
for i in range(max_iterations):
    if i%eval_iters == 0:
        current_time = time.time()
        losses = loss_estimate()

        if i == 0:
            print(f'Step:{i}, training loss: {losses["train"]:.3f}, val loss: {losses["val"]:.3f}')
        else:
            step_duration = current_time - last_eval_time
            print(f'Step:{i}, training loss: {losses["train"]:.3f}, val loss: {losses["val"]:.3f}, time: {step_duration:.1f}s')
        
        last_eval_time = current_time
    # sampling a batch of data
    xb, yb = get_batch('train')

    #computing loss
    logits, loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
toc = time.time()
print(loss.item())
total_time = toc - tic
if total_time < 60:
    print(f"Time: {total_time:.2f}s")
elif total_time < 3600:
    print(f"Time: {total_time/60:.2f}m")
else:
    print(f"Time: {total_time/3600:.2f}h")

Step:0, training loss: 2.281, val loss: 2.362
Step:400, training loss: 2.285, val loss: 2.338, time: 52.5s
Step:800, training loss: 2.250, val loss: 2.277, time: 71.4s
Step:1200, training loss: 2.230, val loss: 2.248, time: 53.4s
Step:1600, training loss: 2.228, val loss: 2.271, time: 47.8s
Step:2000, training loss: 2.199, val loss: 2.219, time: 49.0s
Step:2400, training loss: 2.156, val loss: 2.213, time: 46.3s
Step:2800, training loss: 2.172, val loss: 2.225, time: 51.4s
Step:3200, training loss: 2.139, val loss: 2.129, time: 54.5s
Step:3600, training loss: 2.146, val loss: 2.157, time: 62.6s
Step:4000, training loss: 2.147, val loss: 2.196, time: 64.3s
Step:4400, training loss: 2.117, val loss: 2.184, time: 49.8s
Step:4800, training loss: 2.125, val loss: 2.180, time: 45.2s
2.2932794094085693
Time:675.71s
